# **Notebook entregable**

Incluye:

- Exploración y limpieza de datos.
- Tratamiento de variables financieras y textual
- Ingeniería de características
- Clasificación de gastos
- Análisis financiero
- Entrenamiento y evaluación de modelos
- Métricas de desempeño
- Serialización de modelos (.pkl o .joblib)


# Implementación de motor financiero

## 1.- Recepción de archivo JSON de prueba ya clasificado manualmente 

In [27]:
import json
import random
from pathlib import Path

def cargar_datos_clasificados(carpeta="datos"):
    """
    Selecciona aleatoriamente uno de los archivos JSON clasificados,
    lo convierte a un diccionario y lo devuelve.
    """

    carpeta_json = Path("perfiles_financieros_pruebas_clasificados_para_motor_financiero")

    numero = random.randint(1, 15)

    nombre_archivo = f"{numero:02d}_clasificado_manualmente.json"

    ruta_archivo = carpeta_json / nombre_archivo

    with open(ruta_archivo, "r", encoding="utf-8") as archivo:
        datos = json.load(archivo)

    print(f"Archivo seleccionado: {nombre_archivo}")

    return datos


datos_clasificados = cargar_datos_clasificados()
print (datos_clasificados)

Archivo seleccionado: 14_clasificado_manualmente.json
{'usuario': {'nombre': 'Nicolás Bravo'}, 'periodo': {'inicio': '2026-07-01', 'fin': '2026-07-31'}, 'ingresos': [{'fecha': '2026-07-01', 'descripcion': 'Diseño de sitio web', 'monto': 14000}, {'fecha': '2026-07-02', 'descripcion': 'Proyecto de identidad visual', 'monto': 9500}, {'fecha': '2026-07-03', 'descripcion': 'Consultoría de marketing', 'monto': 7000}, {'fecha': '2026-07-04', 'descripcion': 'Venta de plantillas digitales', 'monto': 3200}], 'transacciones': [{'fecha': '2026-07-02', 'descripcion': 'Renta departamento', 'monto': 7500, 'forma_pago': 'transferencia bancaria', 'clasificacion': ['Consumo'], 'categoria': 'Vivienda', 'grupo': 'Esencial'}, {'fecha': '2026-07-04', 'descripcion': 'Internet Totalplay', 'monto': 699, 'forma_pago': 'tarjeta de crédito', 'tasa_de_interes_de_la_tarjeta': 58, 'clasificacion': ['Consumo', 'Pago de deuda'], 'categoria': 'Otros', 'grupo': 'Esencial'}, {'fecha': '2026-07-05', 'descripcion': 'Adobe 

## 2.- Generación de varibles globales

In [31]:
def calcular_ingreso_mensual(datos_clasificados):
    """
    Calcula el ingreso mensual del usuario.

    Parámetros:
        datos_clasificados (dict): Diccionario con la información del usuario.

    Retorna:
        float: Suma de todos los ingresos registrados durante el periodo.
    """

    ingreso_mensual = 0.0

    for ingreso in datos_clasificados["ingresos"]:
        ingreso_mensual += ingreso["monto"]

    return ingreso_mensual

ingreso_mensual = calcular_ingreso_mensual(datos_clasificados)

print(f"Ingreso mensual: ${ingreso_mensual:,.2f}")

Ingreso mensual: $33,700.00
33700.0


In [32]:
def calcular_consumo_total_mensual(datos_clasificados):
    """
    Calcula el consumo total mensual del usuario.

    Se consideran únicamente las transacciones cuya clasificación
    contenga exclusivamente la etiqueta "Consumo".

    Parámetros:
        datos_clasificados (dict): Diccionario con las transacciones clasificadas.

    Retorna:
        float: Consumo total mensual.
    """

    consumo_total_mensual = 0.0

    for transaccion in datos_clasificados["transacciones"]:

        clasificacion = transaccion["clasificacion"]

        if (
            "Consumo" in clasificacion
            and len(clasificacion) == 1
        ):
            consumo_total_mensual += transaccion["monto"]

    return consumo_total_mensual

consumo_total_mensual = calcular_consumo_total_mensual(datos_clasificados)

print(f"Consumo total mensual: ${consumo_total_mensual:,.2f}")

Consumo total mensual: $12,800.00


In [33]:
def calcular_pago_mensual_de_deudas(datos_clasificados):
    """
    Calcula el pago mensual de deudas del usuario.

    Se consideran todas las transacciones cuya clasificación
    incluya la etiqueta "Pago de deuda", independientemente
    de si también pertenecen a otra clasificación.

    Parámetros:
        datos_clasificados (dict): Diccionario con las transacciones clasificadas.

    Retorna:
        float: Pago mensual de deudas.
    """

    pago_mensual_de_deudas = 0.0

    for transaccion in datos_clasificados["transacciones"]:

        clasificacion = transaccion["clasificacion"]

        if "Pago de deuda" in clasificacion:
            pago_mensual_de_deudas += transaccion["monto"]

    return pago_mensual_de_deudas

pago_mensual_de_deudas = calcular_pago_mensual_de_deudas(datos_clasificados)

print(f"Pago mensual de deudas: ${pago_mensual_de_deudas:,.2f}")

Pago mensual de deudas: $9,914.00


In [34]:
def calcular_ahorro_e_inversion_total(datos_clasificados):
    """
    Calcula el ahorro e inversión total del usuario.

    Se consideran todas las transacciones cuya clasificación
    incluya la etiqueta "Ahorro e inversión".

    Parámetros:
        datos_clasificados (dict): Diccionario con las transacciones clasificadas.

    Retorna:
        float: Ahorro e inversión total.
    """

    ahorro_e_inversion_total = 0.0

    for transaccion in datos_clasificados["transacciones"]:

        clasificacion = transaccion["clasificacion"]

        if "Ahorro e inversión" in clasificacion:
            ahorro_e_inversion_total += transaccion["monto"]

    return ahorro_e_inversion_total

ahorro_e_inversion_total = calcular_ahorro_e_inversion_total(datos_clasificados)

print(f"Ahorro e inversión total: ${ahorro_e_inversion_total:,.2f}")

Ahorro e inversión total: $6,000.00


In [35]:
def calcular_egreso_total(consumo_total_mensual, pago_mensual_de_deudas):
    """
    Calcula el egreso total del usuario.

    El egreso total corresponde a la suma del consumo total mensual
    y el pago mensual de deudas.

    Parámetros:
        consumo_total_mensual (float): Consumo total mensual del usuario.
        pago_mensual_de_deudas (float): Pago mensual de deudas del usuario.

    Retorna:
        float: Egreso total del usuario.
    """

    egreso_total = consumo_total_mensual + pago_mensual_de_deudas

    return egreso_total

egreso_total = calcular_egreso_total(
    consumo_total_mensual,
    pago_mensual_de_deudas
)

print(f"Egreso total: ${egreso_total:,.2f}")

Egreso total: $22,714.00
